#### Kaggle 문제

- https://www.kaggle.com/competitions/dogs-vs-cats-redux-kernels-edition/overview

In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from copy import deepcopy
from PIL import Image
import time
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [45]:
import random

# device 설정
device = 'mps' if torch.mps.is_available() else 'cpu'
print(device)

seed = 42
random.seed(42)
os.environ['PYTHONHASHSEED'] = str(seed) # 해시 시크릿값 고정
np.random.seed(seed)

torch.backends.cudnn.deterministic = True # 확정적 연산 사용 설정
torch.backends.cudnn.benchmark = False # 벤치마크 기능 사용 해제
torch.backends.cudnn.enabled = False # cudnn 기능 사용 해제

if device == 'mps':
    torch.manual_seed(seed)         # CPU 난수
    torch.mps.manual_seed(seed)     # MPS 난수 (CUDA와 동일하게 manual_seed 사용)

mps


In [4]:
import glob

train_dir = './dataset/dogs-vs-cats-redux-kernels-edition/train'
test_dir = './dataset/dogs-vs-cats-redux-kernels-edition/test'

all_train_files = glob.glob(os.path.join(train_dir, '*.jpg'))
all_train_files[0]

'./dataset/dogs-vs-cats-redux-kernels-edition/train/dog.8011.jpg'

In [33]:
all_train_files[0].split('/')[-1][-3:]

'jpg'

In [5]:
img = Image.open('./dataset/dogs-vs-cats-redux-kernels-edition/train/dog.8011.jpg')
img.size

(380, 500)

In [6]:
all_train_files[0].split('/')[-1].split('.')[0]

'dog'

In [7]:
all_train_files = glob.glob(os.path.join(train_dir, '*.jpg'))
test_list = glob.glob(os.path.join(test_dir, "*.jpg"))
train_labels = [ path.split('/')[-1].split('.')[0] for path in all_train_files]
train_list, val_list = train_test_split(all_train_files, test_size=0.1, stratify=train_labels, random_state=seed)
print(len(train_list), len(val_list))

22500 2500


In [37]:
input_size = 224
transforms_for_train = transforms.Compose([
    transforms.RandomResizedCrop(input_size, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])
transforms_for_val_test = transforms.Compose([
    transforms.Resize(input_size),
    transforms.CenterCrop(input_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])

class CustomImageDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform
        
    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, index):
        img_path = self.file_list[index]
        if img_path.split('/')[-1][-3:] == 'jpg':
            img = Image.open(img_path)
            if self.transform is not None:
                img_transform = self.transform(img)
                label = img_path.split('/')[-1].split('.')[0]
                if label == 'dog':
                    label = 1
                elif label == 'cat':
                    label = 0
        return img_transform, label
    
dataset_train = CustomImageDataset(train_list, transform=transforms_for_train)
dataset_valid = CustomImageDataset(val_list, transform=transforms_for_val_test)
dataset_test = CustomImageDataset(test_list, transform=transforms_for_val_test)


train_batches = DataLoader(dataset=dataset_train, batch_size=8, shuffle=True)
valid_batches = DataLoader(dataset=dataset_valid, batch_size=8, shuffle=False)
test_batches = DataLoader(dataset=dataset_test, batch_size=8, shuffle=False)


In [38]:
import timm

model = timm.create_model('vit_base_patch32_224_in21k',pretrained=True)

In [46]:
model.head = nn.Sequential(
    nn.Linear(768, 21843, bias=True),
    nn.LeakyReLU(),
    nn.BatchNorm2d(21843),
    nn.Linear(21843, 512, bias=True),
    nn.LeakyReLU(),
    nn.BatchNorm2d(512),
    nn.Linear(512, 1, bias=True),
    nn.Sigmoid()
)
model.to(device) # GPU
loss_func = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters())

